___

# <font color= #8C6B4F> **GANs for Map Generation: Report** </font>
#### <font color= #2E9AFE> `Deep Learning`</font>
<Strong> Sofía Maldonado, Oscar Josué Rocha & Viviana Toledo </Strong>

_07/04/2026._

___

## <font color= #C2A878> **Objective** </font>

This project aims to build a Generative Adversarial Network (GAN) for map generation via style transfer. Using old and fantasy map images as the style, the GAN should be able to **convert satellite images into specific map representations**.

## <font color= #C2A878> **Data** </font>

Multiple datasets were required for the construction of this project, as well as for trial purposes, since the dataset was not crafted nor achieved on the first try. Our aim is to build a style transfer GAN, which needs a training dataset and a style dataset. The methodology and source of each dataset is detailed below.

### <font color= #F5ECD9> &nbsp; • **Satellite Images** </font>

A first set of satellite images was obtained from the `Sentinel2 satellite`, which is an **Earth observation mission** from the Copernicus Programme by the European Space Agency (ESA). It was first launched on June 2015, and the last launch was performed on September 2024. It orbits at an **altitude of 786 km** and a **maximum Ground Sample Distance (GSD) of 60 meters** (distance between pixels), which results in high-context images [1]. 

The script written for downloading Sentinel2 images can be found in [`scripts/sentinel2.py`](../scripts/sentinel2.py). The dataset contains **5,332 images**.

<p align="center">
  <img src="../reports/figures/1.1-data.png" alt="Sentinel2" width="300">
</p>

The second set of satellite images was obtained from `Open Aerial Map (OAM)`, which is an open collection of aerial imagery. Its objective is to _"host and provide access to imagery for humanitarian response and disaster preparedness"_ [2]. OAM is a colaborative effort, and contains a **variety of satellite images related to areas affected by disasters**, resulting in **close-up images** of certain geographical areas. Although the images were more suited for the project, clouds obstructed some views, not all images have a proper quality-control, and some of them are really zoomed-in.

The script written for downloading OAM images can be found in [`scripts/open_aerial_map.py`](../scripts/open_aerial_map.py). The dataset contains **3,237 images**.

<p align="center">
    <img src="../reports/figures/1.2-data.png" alt="Open Aerial Map" width="300">
</p>

Finally, the last set of images was obtained from `Google Maps` satellite view: a **mosaic of aerial photographies at variable altitudes to map and archive the world at large**. A collection of images from 52 countries across the 5 continents were screenshoted, taken from the 3 most populated cities of each country. Although Google's satellite images can be extracted from an API, the API is blocked behind a paywall, limiting downloads and open access to data. For this reason, a more manual approach was taken for data extraction.

The dataset contains **2,697 images**, and was too large to be uploaded.

<p align="center">
    <img src="../reports/figures/1.3-data.png" alt="Google Maps Satellite View" width="300">
</p>

### <font color= #F5ECD9> &nbsp; • **Map Styles** </font>

Of the 3 datasets obtained, each was paired with a specific style suited for its image range. The Sentinel satellite was paired with fantasy maps at large scale, OAM was paired with old maps, and Google Maps' imagery was paired with close-up fantasy city maps. Each **set of styles contained 20 images**, which were **handpicked on Google Images**.

<p align="center">
  <span style="display: inline-block; text-align: center; margin-right: 60px;">
    <img src="../reports/figures/1.4-data.png" height="300" />
    <br/>
    <em>Style for Sentinel's images</em>
  </span>

  <span style="display: inline-block; text-align: center; margin-right: 60px;">
    <img src="../reports/figures/1.5-data.webp" height="300" />
    <br/>
    <em>Style for OAM's images</em>
  </span>

  <span style="display: inline-block; text-align: center;">
    <img src="../reports/figures/1.6-data.png" height="300" />
    <br/>
    <em>Style for Google Maps' images</em>
  </span>
</p>

## <font color= #C2A878> **Modeling – CycleGAN** </font>

A `CycleGAN` was chosen for the task at hand because it learns from **unpaired data**, meaning there's no information provided as to which image from domain _A_ matches which from domain _B_. Instead, the model `learns to translate between two domains by capturing their underlying distributions and assuming there's a relationship between them`.

The model defines two mappings, such that transformations from $G_A: A \rightarrow B$ and $G_B: B \rightarrow A$ should **approximately reconstruct the original image**. In essence, `it maps an image to itself via an intermediate representation`, or a translation to another domain:

$$
a \rightarrow G_A(a) \rightarrow G_B(G_A(a)) \approx a
$$

CycleGAN implements a **_cyle consistency loss_** as well as an **_adversarial loss_** between image translations to ensure quality. The former prevents the learned mappings $G_A$ and $G_B$ from contradicting each other (the translations), while the latter represents the Discriminators $D_A$ and $D_B$, tasked with ensuring that the generated images are as close to the original distribution as possible. Such that the model's overall loss function is defined as follows [3]: 

$$
\begin{aligned}

\mathcal{L} (G_A, G_B, D_A, D_B) = 
& \mathcal{L}_{\mathrm{GAN}} (G_A, D_B, A, B) \\
& + \mathcal{L}_{\mathrm{GAN}} (G_B, D_A, B, A) \\
& + \lambda \mathcal{L}_{\mathrm{cyc}} (G_A, G_B) \\

\end{aligned}
$$

where:

- The first loss function evaluates transformations $G_A:A \rightarrow B,$
- The second loss function evaluates transformations $G_B:B \rightarrow A,$
- And the third loss function enforces **cycle consistency**, so that mappings don't contradict each other.

Taking into account that our datasets and styles are not paired between each other, and our objective is to **translate images from one domain (satellite images) to another (map representations)**, we imported the CycleGAN model proposed by the authors of the aforementioned paper and creators of the model, available in their GitHub repository [4]. On its own, CycleGAN is incredibly powerful to learn unpaired image representations. Thus, we **trained 3 models for each style using 5 epochs**, achieving great results.

## <font color= #C2A878> **Results** </font>

For exploratory and practicality reasons, **two branches were created to store the models**, and will remain there for isolation. The model for OAM data was stored in `poder_computacional` [`./maps_test`](https://github.com/ch0fas/gans_for_maps/tree/poder_computacional/checkpoints/maps_test), while Google Maps and Sentinel2 models' were stored on `gl_imgs`, under the names [`./maps_test`](https://github.com/ch0fas/gans_for_maps/tree/gl_imgs/checkpoints/maps_test) and [`./maps_test_2`](https://github.com/ch0fas/gans_for_maps/tree/gl_imgs/checkpoints/maps_test_2), respectively.

To compare the results from our 3 models, we used two separate metrics. First, we used *Fréchet Inception Distance* **(FID)** which is defined by the following formula [4]:
$$
\text{FID} = \left\| \mu - \mu_w \right\|_2^2 + \text{Tr}\left( \Sigma + \Sigma_w - 2\left( \Sigma \Sigma_w \right)^{1/2} \right)
$$
This is a standard metric used to measure quality in the results of generative models, including GANs.

Besides FID, we also created a **custom-made metric** for generative models. We call it ***YAGM***, short for **Y**et **A**nother **G**AN **M**etric. It is defined below:
$$
\text{YAGM} =  |(||(x_{real}-\tilde{x_{real}})||-||(x_{fake}-\tilde{x_{fake}})||)| 
$$

To test the results, we generated 20 random images with each of the 3 models and calculated their FID and YAGM values. The scripts used to generate the images and perform the tests can be found in the `scripts` folders in the `gl_imgs` and `poder_computacional` branches.

### <font color= #F5ECD9> &nbsp; • **Sentinel2** </font>

The model trained for Sentinel2 centers around **large-scale images with more landscape context**, creating zoomed-out maps that represent the overall geographical shape of an image. However, it suffers from hallucinations in order to achieve the "island look" a lot of fantasy maps have in common:  

<p align="center">
    <img src="../reports/figures/2.1-results.png" width="370">
    <img src="../reports/figures/2.2-results.png" width="370">
    <img src="../reports/figures/2.3-results.png" width="370">
    <img src="../reports/figures/2.4-results.png" width="370">
</p>

The model's scores are as follow:
- **FID**: 280.74
- **YAGM**: 1872.92
- **YAGM Standard Deviation (STD)**: 1441.16

THis model had decent results, as the original images were very simple (just simple fields) and were therefore easy for the model to comprehend. The style images were slightly complex detail-wise which added complexity to what the model was trying to achieve, and were taken from different angles (satellite images are usually seen straight-down, while fantasy maps are drawn with a more horizontal perspective).

### <font color= #F5ECD9> &nbsp; • **OAM** </font>

OAM's model centers around **_close-up images with not many details_**, creating maps that capture the distribution of cities, with an old worn-out look:

<p align="center">
    <img src="../reports/figures/2.5-results.png" width="370">
    <img src="../reports/figures/2.6-results.png" width="370">
    <img src="../reports/figures/2.7-results.png" width="370">
</p>

Our OAM model had the following scores:
- **FID**: 241.74
- **YAGM**: 1497.74
- **YAGM Standard Deviation (STD)**: 871.54

This was by far our best model of the bunch metrics-wise. This is understandable, as the style images and original source images had very little detail to learn, and were easy to reconstruct with a GAN network. They are both taken from similar vantage-points too, which made it easier for the model to comprehend.

### <font color= #F5ECD9> &nbsp; • **Google Maps** </font>

Finally, Google Maps' model focuses on **close-up images with detail**, giving a more polished look to the maps and allowing for greater stylization:

<p align="center">
    <img src="../reports/figures/2.8-results.png" width="370">
    <img src="../reports/figures/2.9-results.png" width="370">
    <img src="../reports/figures/2.10-results.png" width="370">
    <img src="../reports/figures/2.11-results.png" width="370">
</p>

This model had the following scores:
- **FID**: 304.38
- **YAGM**: 2404.69
- **YAGM Standard Deviation (STD)**: 2146.52

This was the model that had the worst scores, in both FID and YAGM. The images, for both content and style, were far more complex than the images used for the other two examples, which made comprehending their features a more complicated task for the computer. Besides, our old maps were almost exclusively from european cities in the middle ages and early industrial period. Today, cities in europe and the rest of the world have developed very differently, which makes the comparison between them slightly more alien than expected. Also, the standard deviation for YAGM was considerably closer to the mean than in the other two models, which exemplifies the big variety of results here.

### <font color= #F5ECD9> &nbsp; • **General Conclusions** </font>

Our custom YAGM metric, like FID, goes lower with better quality results. Unlike FID, however, YAGM is slightly more sensible to "worse" results, and rises considerably faster. A value of 240 in FID translates to around 1500 in YAGM, but a value of 304 FID, a 25% increase, results in a YAGM of approximately 2400, a 60% increase. This is not linear: a 280 in FID, a difference of around 16%, results in a 20% higher YAGM. 

In conclusion, YAGM is more punishing of lesser quality than FID. YAGM could therefore be used as a performance metric where good numerical results are more critical, such as with computer vision where results are merely judged by other computers and not by humans.

Of course, these are all numerical metrics, and models can be judged by the user based on how humans interpret their results. For our part, we thought the Google Maps model was the most impressive, even if it had the lowest numerical results out of the three.

## <font color= #C2A878> **Challenges** </font>

The main challenge we faced while developing this project was **finding suitable images and styles that matched our vision**, since there are multiple approaches you can take for _"converting satellite images to map representations"_, as showcased in the results section. Partly, this is the reason why we have 3 different datasets, each with their own style. On the modeling front, the code for the CycleGAN [5] was fairly easy to comprehend and implement, to the point no changes needed to be made to get it working.

## <font color= #C2A878> **Applications** </font>

The main field of applications that we suggest for these models are **creative concepts**. These images do not generate production-ready content, but can be used to inspire artists on the visual outlook of a fantasy world or any other map that they want to produce with a unique style. It helps that we have three models: it showcases different styles people might be interested in, widening appeal.

For our part, we have made a small demo in **HuggingFace** that allows the user to access any of the 3 models remotely and use all three on a single image. It can be accessed [here](https://huggingface.co/spaces/chofas/maps_with_gans). The source code for the app can be found [here](https://huggingface.co/spaces/chofas/maps_with_gans/tree/main).

## <font color= #C2A878> **Bibliography and References** </font>

[1] Sentinel-2. (2026, 1 de febrero). _Wikipedia_. https://en.wikipedia.org/wiki/Sentinel-2

[2] Open Aerial Map Docs. (2021, 2 de julio). _OpenAerialMap Ecosystem_. https://docs.openaerialmap.org/ecosystem/

[3] Zhu, J.-Y., Park T., Isola, P. & Efros, A. A. (2017). _Unpaired Image-to-Image Translation using Cycle-Consistent Adversarial Networks_. https://arxiv.org/pdf/1703.10593

[4] Heusel, M., Ramsauer, H., Unterthiner, T & Nessler, B. (2017) *GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium*. https://arxiv.org/pdf/1706.08500 

[5] Zhu, J.-Y., Park T., Isola, P. & Efros, A. A. (2025). _pytorch-CycleGAN-and-pix2pix_. GitHub. https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix